# Analisis tambahan: kalibrasi, *decision curve*, uji terkoreksi, dan proksi usia
Menghasilkan angka pada Subbab III.B–C naskah: Brier skill score, *intercept*/*slope* kalibrasi, *decision curve analysis* (Gbr. 3), *corrected resampled t-test* Nadeau–Bengio, dan hubungan jenis pekerjaan dengan usia.

Prasyarat: `01_eksperimen_utama.ipynb` dan `02_revisi_M1_M3.ipynb` sudah dijalankan (memakai `results/eksp3b_best_params.json`, `eksp2_per_fold.csv`, `eksp4_smotenc_per_fold.csv`).

In [ ]:
import os, json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.ticker as mt
from scipy import stats
from sklearn.compose import ColumnTransformer; from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer; from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
DATA_DIR = os.path.join("..", "data"); OUT = os.path.join("..", "results"); SEED = 42
df = pd.read_csv(os.path.join(DATA_DIR, "healthcare-dataset-stroke-data.csv")).drop(columns="id")
df = df[df.gender != "Other"].reset_index(drop=True); y = df.pop("stroke").values; X = df
NUM = ["age", "avg_glucose_level", "bmi"]; BIN = ["hypertension", "heart_disease"]; CAT = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]
CATS = [sorted(X[c].unique()) for c in CAT]
prep = lambda: ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM), ("bin", "passthrough", BIN),
                                  ("cat", OneHotEncoder(categories=CATS, drop="first", handle_unknown="ignore", sparse_output=False), CAT)])
xtr, xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

## 1. Rekonstruksi prediksi data uji (HGB terpilih dan RF *class weight* pembanding)

In [ ]:
hp = json.load(open(os.path.join(OUT, "eksp3b_best_params.json")))
conv = lambda v: None if v == "None" else (float(v) if "." in v else int(v))
hgb = HistGradientBoostingClassifier(random_state=SEED, **{k.split("__")[-1]: conv(v) for k, v in hp.items()})
p_h = Pipeline([("p", prep()), ("m", hgb)]).fit(xtr, ytr).predict_proba(xte)[:, 1]
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=SEED, max_depth=5, min_samples_leaf=10, max_features=0.5)
p_r = Pipeline([("p", prep()), ("m", rf)]).fit(xtr, ytr).predict_proba(xte)[:, 1]
for nm, p in [("HGB tanpa penanganan", p_h), ("RF class weight", p_r)]:
    print(f"{nm}: ROC-AUC {roc_auc_score(yte, p):.3f} | PR-AUC {average_precision_score(yte, p):.3f} | Brier {brier_score_loss(yte, p):.4f}")

## 2. Brier skill score dan *intercept*/*slope* kalibrasi (IK 95% *bootstrap* 2.000 kali)

In [ ]:
def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def fit_logit(Xd, yy, offset=None, it=50):
    b = np.zeros(Xd.shape[1]); off = np.zeros(len(yy)) if offset is None else offset
    for _ in range(it):
        mu = 1 / (1 + np.exp(-(Xd @ b + off))); W = mu * (1 - mu)
        b = b + np.linalg.solve(Xd.T @ (Xd * W[:, None]), Xd.T @ (yy - mu))
    return b
def calib(p, yy):
    lp = logit(p); a = fit_logit(np.ones((len(yy), 1)), yy, offset=lp)[0]
    return a, fit_logit(np.c_[np.ones(len(yy)), lp], yy)[1]
prev = yte.mean(); null = prev * (1 - prev); rows = []
for nm, p in [("HGB tanpa penanganan", p_h), ("RF class weight", p_r)]:
    a, s = calib(p, yte); bs = brier_score_loss(yte, p); r = np.random.RandomState(SEED); A, S, B = [], [], []
    for _ in range(2000):
        i = r.randint(0, len(yte), len(yte))
        if yte[i].sum() == 0: continue
        aa, ss = calib(p[i], yte[i]); A.append(aa); S.append(ss); B.append(1 - brier_score_loss(yte[i], p[i]) / (yte[i].mean() * (1 - yte[i].mean())))
    ci = lambda v: "-".join(f"{x:.3f}" for x in np.percentile(v, [2.5, 97.5]))
    rows.append(dict(Model=nm, mean_p=p.mean(), observed=prev, intercept=a, intercept_CI=ci(A), slope=s, slope_CI=ci(S), Brier=bs, Brier_nol=null, BSS=1 - bs / null, BSS_CI=ci(B), p_max=p.max()))
cal = pd.DataFrame(rows); cal.to_csv(os.path.join(OUT, "stat_kalibrasi_lanjut.csv"), index=False); display(cal.round(3).T)
e2 = pd.concat([pd.read_csv(os.path.join(OUT, "eksp2_nested_cv.csv")), pd.read_csv(os.path.join(OUT, "eksp4_smotenc.csv"))])
nullp = y.mean() * (1 - y.mean()); t = e2[e2.Mode == "tuned"][["Strategi", "Model", "Brier"]].copy(); t["BSS"] = 1 - t.Brier / nullp
print(f"Brier model nol (5.109 data) = {nullp:.4f}"); display(t.round(3))

## 3. *Decision curve analysis* (Gbr. 3)

In [ ]:
n = len(yte); T = np.linspace(0.005, 0.25, 200)
nb = np.array([((p_h >= t) & (yte == 1)).sum() / n - ((p_h >= t) & (yte == 0)).sum() / n * t / (1 - t) for t in T]); na = prev - (1 - prev) * T / (1 - T)
ok = T[nb > np.maximum(na, 0) + 0.0005]; print(f"Net benefit model > alternatif pada ambang {ok.min():.3f}-{ok.max():.3f}")
t0 = 0.064; nb0 = ((p_h >= t0) & (yte == 1)).sum() / n - ((p_h >= t0) & (yte == 0)).sum() / n * t0 / (1 - t0); na0 = prev - (1 - prev) * t0 / (1 - t0)
print(f"Ambang {t0}: NB model {nb0:.4f} | NB tangani semua {na0:.4f} | pengurangan intervensi tidak perlu per 100: {(nb0 - na0) / (t0 / (1 - t0)) * 100:.1f}")
pd.DataFrame(dict(ambang=T, NB_model=nb, NB_semua=na)).to_csv(os.path.join(OUT, "dca_hgb.csv"), index=False)
fig, ax = plt.subplots(figsize=(5.2, 3.9))
ax.plot(T, nb, color="#1f5fa8", lw=1.8, label="HGB tanpa penanganan"); ax.plot(T, na, color="#8c8c8c", lw=1.2, ls="--", label="Tangani semua pasien")
ax.axhline(0, color="k", lw=0.9, label="Tidak menangani pasien"); ax.axvline(t0, color="#1f5fa8", lw=0.8, ls=":")
ax.text(0.067, 0.036, "ambang Youden\n0,064", fontsize=7.5, color="#1f5fa8", va="top"); ax.set_xlim(0, 0.25); ax.set_ylim(-0.02, 0.055)
ax.set_xlabel("Ambang probabilitas"); ax.set_ylabel("Net benefit"); fmt = mt.FuncFormatter(lambda v, _: f"{v:.2f}".replace(".", ","))
ax.xaxis.set_major_formatter(fmt); ax.yaxis.set_major_formatter(fmt); ax.legend(fontsize=7.5, loc="upper right", frameon=False)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "gambar_dca.png"), dpi=300); plt.show()

## 4. *Corrected resampled t-test* Nadeau–Bengio (10 *fold*, rasio data uji/latih 0,25)

In [ ]:
fr = pd.concat([pd.read_csv(os.path.join(OUT, "eksp2_per_fold.csv")), pd.read_csv(os.path.join(OUT, "eksp4_smotenc_per_fold.csv"))])
get = lambda s, m, mo: fr[(fr.Strategi == s) & (fr.Model == m) & (fr.Mode == mo)].sort_values("Fold").PR_AUC.values
def nadeau_bengio(d, k=10, ratio=0.25):
    d = np.asarray(d); t = d.mean() / np.sqrt((1 / k + ratio) * d.var(ddof=1)); return t, 2 * stats.t.sf(abs(t), k - 1)
rows = []
for m in ["RF", "HGB", "LR"]:
    for s in ["none", "smote", "smotenc", "rus", "class_weight"]:
        d = get(s, m, "tuned") - get(s, m, "default"); t, p = nadeau_bengio(d); rows.append(dict(Uji="tuning vs bawaan", Model=m, Strategi=s, mean_delta=d.mean(), t=t, p=p))
for m in ["RF", "HGB", "LR"]:
    d = get("smotenc", m, "tuned") - get("smote", m, "tuned"); t, p = nadeau_bengio(d); rows.append(dict(Uji="SMOTE-NC vs SMOTE", Model=m, Strategi="", mean_delta=d.mean(), t=t, p=p))
d = get("class_weight", "RF", "tuned") - get("none", "LR", "default"); t, p = nadeau_bengio(d); rows.append(dict(Uji="RF class weight vs LR tanpa penanganan", Model="", Strategi="", mean_delta=d.mean(), t=t, p=p))
nbt = pd.DataFrame(rows); nbt.to_csv(os.path.join(OUT, "stat_nadeau_bengio.csv"), index=False); display(nbt.round(4))

## 5. Jenis pekerjaan sebagai proksi usia

In [ ]:
g = X.assign(stroke=y).groupby("work_type").agg(n=("stroke", "size"), stroke=("stroke", "sum"), usia_median=("age", "median"), usia_min=("age", "min"), usia_max=("age", "max"))
g["laju_stroke_%"] = (g.stroke / g.n * 100).round(2); g.to_csv(os.path.join(OUT, "stat_work_type_usia.csv")); display(g)